# 🧠 EDiTH — Train DR Classifier
**Fast Training (~10-15 min) for EfficientNet-B0 on APTOS 2019**

### Before you start:
1. `Runtime → Change runtime type → T4 GPU`
2. Get your Kaggle API key: [kaggle.com/settings](https://www.kaggle.com/settings) → API → **Create New Token** (downloads `kaggle.json`)
3. `Runtime → Run all`
4. When prompted, **upload your `kaggle.json` file**
5. Wait ~10-15 minutes (with AMP & EfficientNet-B0) → download trained weights at the end

## Step 1: Check GPU

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

GPU Available: True
GPU: Tesla T4
Memory: 15.6 GB


## Step 2: Install Dependencies

In [ ]:
!pip install -q timm scikit-learn pandas kaggle

## Step 3: Upload Kaggle API Key & Download Dataset

**Run the cell below.** It will ask you to upload your `kaggle.json` file.

If you don't have it:
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to **API** section
3. Click **Create New Token**
4. It downloads a `kaggle.json` file — upload that below

In [ ]:
import os
from google.colab import files

# Upload kaggle.json
print("📤 Please upload your kaggle.json file:")
uploaded = files.upload()

# Set up Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print("✅ Kaggle credentials configured!")

📤 Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
✅ Kaggle credentials configured!


In [ ]:
# Download and extract APTOS 2019 dataset
!kaggle competitions download -c aptos2019-blindness-detection -p ./aptos_data
print("\n📦 Extracting...")
!cd aptos_data && unzip -qo aptos2019-blindness-detection.zip
print("✅ Dataset ready!")

100% 9.51G/9.51G [01:14<00:00, 136MB/s]


📦 Extracting...
✅ Dataset ready!


In [ ]:
import os
DATA_DIR = './aptos_data'
IMAGE_DIR = os.path.join(DATA_DIR, 'train_images')
print("Files in data dir:", os.listdir(DATA_DIR))
num_images = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')])
print(f"Training images found: {num_images}")

Files in data dir: ['test_images', 'train.csv', 'sample_submission.csv', 'aptos2019-blindness-detection.zip', 'train_images', 'test.csv']
Training images found: 3662


## Step 4: Prepare Data

In [ ]:
import pandas as pd
import numpy as np
import cv2
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print(f"Total images: {len(df)}")
print(f"\nClass distribution:")
for grade, count in df['diagnosis'].value_counts().sort_index().items():
    labels = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
    print(f"  Grade {grade} ({labels[grade]}): {count}")

Total images: 3662

Class distribution:
  Grade 0 (No DR): 1805
  Grade 1 (Mild): 370
  Grade 2 (Moderate): 999
  Grade 3 (Severe): 193
  Grade 4 (Proliferative): 295


In [ ]:
# ---- Preprocessing functions ----

def crop_fundus(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    pad = 10
    x, y = max(0, x - pad), max(0, y - pad)
    w = min(image.shape[1] - x, w + 2 * pad)
    h = min(image.shape[0] - y, h + 2 * pad)
    return image[y:y+h, x:x+w]

def ben_graham(image, size=224):
    image = crop_fundus(image)
    image = cv2.resize(image, (size, size))
    image = image.astype(np.float32)
    blur = cv2.GaussianBlur(image, (0, 0), size / 30.0)
    image = cv2.addWeighted(image, 4, blur, -4, 128)
    return np.clip(image, 0, 255).astype(np.uint8)

def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

print("✅ Preprocessing functions ready")

✅ Preprocessing functions ready


In [ ]:
# ---- Dataset class ----

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

class APTOSDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, size=224):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.size = size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.image_dir, f"{row['id_code']}.png")
        image = cv2.imread(path)
        if image is None:
            image = np.zeros((self.size, self.size, 3), dtype=np.uint8)
        image = ben_graham(image, self.size)
        image = apply_clahe(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        return image, int(row['diagnosis'])

# Split data 80/20
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(df, df['diagnosis']))
train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Datasets & weighted sampler for class imbalance
train_ds = APTOSDataset(train_df, IMAGE_DIR, train_tf)
val_ds = APTOSDataset(val_df, IMAGE_DIR, val_tf)

class_counts = train_df['diagnosis'].value_counts().sort_index().values.astype(float)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[l] for l in train_df['diagnosis'].values]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Data ready — {len(train_loader)} train batches, {len(val_loader)} val batches")

Train: 2929 | Val: 733
✅ Data ready — 92 train batches, 23 val batches


## Step 5: Build Model & Train

In [ ]:
import timm
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report, roc_auc_score
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Focal Loss (handles class imbalance) ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean()

# --- Model ---
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)

alpha = torch.tensor(class_weights / class_weights.sum(), dtype=torch.float32).to(device)
criterion = FocalLoss(alpha=alpha, gamma=2.0)

# Differential learning rates: lower for pretrained backbone, higher for classifier head
backbone = [p for n, p in model.named_parameters() if 'classifier' not in n]
head = [p for n, p in model.named_parameters() if 'classifier' in n]
optimizer = optim.AdamW([
    {'params': backbone, 'lr': 3e-5},
    {'params': head, 'lr': 3e-4},
], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ EfficientNet-B0 ready: {total_params:.1f}M parameters")

Using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

✅ EfficientNet-B0 ready: 4.0M parameters


In [ ]:
# ========== FAST TRAINING LOOP WITH AMP & EARLY STOPPING ==========

EPOCHS = 10
best_qwk = 0.0
patience = 5
patience_counter = 0
DR_LABELS = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# Mixed precision scaler for 2x faster GPU training
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

print('=' * 65)
print('🚀 Starting fast training with AMP — takes ~10-15 mins on T4 GPU')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    # --- Train ---
    model.train()
    train_loss, correct, total = 0, 0, 0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                out = model(images)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()

    train_acc = correct / total

    # --- Validate ---
    model.eval()
    val_preds, val_labels_list, val_probs = [], [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            probs = torch.softmax(out, dim=1)
            _, pred = out.max(1)
            val_preds.extend(pred.cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())
            val_probs.extend(probs.cpu().numpy())

    scheduler.step()

    # --- Metrics ---
    qwk = cohen_kappa_score(val_labels_list, val_preds, weights='quadratic')
    val_acc = (np.array(val_preds) == np.array(val_labels_list)).mean()
    elapsed = time.time() - start

    star = ''
    if qwk > best_qwk:
        best_qwk = qwk
        patience_counter = 0
        torch.save(model.state_dict(), 'efficientnet_b0_dr.pth')
        torch.save(model.state_dict(), 'efficientnet_b3_dr.pth')  # fallback compatibility
        star = '  ★ NEW BEST — saved!'
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'\n✋ Early stopping triggered at epoch {epoch} (no improvement for {patience} epochs)')
            break

    print(f'Epoch {epoch:2d}/{EPOCHS} | {elapsed:.0f}s | '
          f'Train Acc: {train_acc:.3f} | '
          f'Val Acc: {val_acc:.3f} | '
          f'QWK: {qwk:.4f}{star}')

print(f'\n{"=" * 65}')
print(f'✅ Training complete! Best QWK: {best_qwk:.4f}')
print(f'{"=" * 65}')


🚀 Starting fast training with AMP — takes ~10-15 mins on T4 GPU
Epoch  1/10 | 359s | Train Acc: 0.505 | Val Acc: 0.578 | QWK: 0.7053  ★ NEW BEST — saved!
Epoch  2/10 | 346s | Train Acc: 0.548 | Val Acc: 0.593 | QWK: 0.7318  ★ NEW BEST — saved!
Epoch  3/10 | 348s | Train Acc: 0.590 | Val Acc: 0.613 | QWK: 0.7416  ★ NEW BEST — saved!
Epoch  4/10 | 351s | Train Acc: 0.605 | Val Acc: 0.611 | QWK: 0.7486  ★ NEW BEST — saved!
Epoch  5/10 | 350s | Train Acc: 0.623 | Val Acc: 0.600 | QWK: 0.7392
Epoch  6/10 | 345s | Train Acc: 0.630 | Val Acc: 0.608 | QWK: 0.7481
Epoch  7/10 | 362s | Train Acc: 0.655 | Val Acc: 0.629 | QWK: 0.7610  ★ NEW BEST — saved!
Epoch  8/10 | 357s | Train Acc: 0.662 | Val Acc: 0.610 | QWK: 0.7511
Epoch  9/10 | 351s | Train Acc: 0.663 | Val Acc: 0.634 | QWK: 0.7660  ★ NEW BEST — saved!
Epoch 10/10 | 353s | Train Acc: 0.677 | Val Acc: 0.619 | QWK: 0.7736  ★ NEW BEST — saved!

✅ Training complete! Best QWK: 0.7736


## Step 6: Final Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('efficientnet_b0_dr.pth', map_location=device, weights_only=True))
model.eval()

val_preds, val_labels_list, val_probs = [], [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        out = model(images)
        probs = torch.softmax(out, dim=1)
        _, pred = out.max(1)
        val_preds.extend(pred.cpu().numpy())
        val_labels_list.extend(labels.cpu().numpy())
        val_probs.extend(probs.cpu().numpy())

qwk = cohen_kappa_score(val_labels_list, val_preds, weights='quadratic')
acc = (np.array(val_preds) == np.array(val_labels_list)).mean()

# Sensitivity & Specificity
cm = confusion_matrix(val_labels_list, val_preds)
sensitivities = []
specificities = []
for i in range(5):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivities.append(sens)
    specificities.append(spec)

print("📊 FINAL METRICS")
print("=" * 50)
print(f"  Quadratic Weighted Kappa (QWK): {qwk:.4f}")
print(f"  Accuracy:                       {acc:.4f}")
print(f"  Sensitivity (macro):            {np.mean(sensitivities):.4f}")
print(f"  Specificity (macro):            {np.mean(specificities):.4f}")

try:
    auc = roc_auc_score(val_labels_list, val_probs, multi_class='ovr', average='macro')
    print(f"  AUC-ROC (macro):                {auc:.4f}")
except Exception:
    pass

print(f"\n📋 Per-Class Report:")
print(classification_report(val_labels_list, val_preds, target_names=DR_LABELS))

print(f"\n🔢 Confusion Matrix:")
print(f"{'':>15} Predicted →")
print(f"{'Actual ↓':>15}  {'  '.join(f'G{i}' for i in range(5))}")
for i, row in enumerate(cm):
    print(f"  Grade {i} ({DR_LABELS[i]:>13}): {row}")

print(f"\n✅ Model file size: {os.path.getsize('efficientnet_b0_dr.pth') / 1e6:.1f} MB")

📊 FINAL METRICS
  Quadratic Weighted Kappa (QWK): 0.7736
  Accuracy:                       0.6194
  Sensitivity (macro):            0.5223
  Specificity (macro):            0.9111
  AUC-ROC (macro):                0.8669

📋 Per-Class Report:
               precision    recall  f1-score   support

        No DR       0.94      0.88      0.91       361
         Mild       0.30      0.50      0.38        74
     Moderate       0.71      0.26      0.38       200
       Severe       0.17      0.44      0.24        39
Proliferative       0.31      0.54      0.40        59

     accuracy                           0.62       733
    macro avg       0.49      0.52      0.46       733
 weighted avg       0.72      0.62      0.63       733


🔢 Confusion Matrix:
                Predicted →
       Actual ↓  G0  G1  G2  G3  G4
  Grade 0 (        No DR): [317  32   5   3   4]
  Grade 1 (         Mild): [13 37 10  8  6]
  Grade 2 (     Moderate): [ 5 46 51 55 43]
  Grade 3 (       Severe): [ 1  2  2 1

## Step 7: Download Trained Model

Run the cell below → click the download link.

**Then place the file at:** `EDiTH/backend/models/weights/efficientnet_b0_dr.pth`

In [ ]:
from google.colab import files
print("📥 Downloading trained model...")
print("\n📂 Place this file at:")
print("   EDiTH/backend/models/weights/efficientnet_b0_dr.pth")
print("\nThen restart the EDiTH backend server.")
files.download('efficientnet_b0_dr.pth')

📥 Downloading trained model...

📂 Place this file at:
   EDiTH/backend/models/weights/efficientnet_b0_dr.pth

Then restart the EDiTH backend server.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>